In [ ]:
import os
import json
import gdown
import itertools

import numpy as np
import pandas as pd
import seaborn as sns
import pyspark.pandas as ps
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler
)


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

spark = (
    SparkSession.builder.appName("AnaliseDengue")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

In [2]:
files_id = ["1Rph--dtL4aF_CwWrJIS_YaRN0Sa7z9lL", "1WXvMMrBsmnz8Z2x68fpS0T-o2d1qZxJ_", "1IY0fqEHXum51sTRzfpoLpP1x6A9USz1w"]
file_names = [24, 25, 26]
for file_id, file_name in zip(files_id, file_names):
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        f"data/DENGBR{file_name}.csv",
        quiet=False
    )

df = spark.read.csv(
    "data/DEN*.csv",
    header=True,
    inferSchema=True
)

df = df.withColumn(
    'SG_UF_NOT', 
    F.upper(F.col('SG_UF_NOT').cast('string'))
)

df = df.drop("ID_AGRAVO", "ID_REGIONA", "ID_UNIDADE", "ID_RG_RESI", "ID_OCUPA_N", "ID_PAIS")

df.write.mode("overwrite").parquet("data/dengue.parquet")

Downloading...
From (original): https://drive.google.com/uc?id=1Rph--dtL4aF_CwWrJIS_YaRN0Sa7z9lL
From (redirected): https://drive.google.com/uc?id=1Rph--dtL4aF_CwWrJIS_YaRN0Sa7z9lL&confirm=t&uuid=e7d356fc-7aa5-4a6f-a6b9-148fc246d480
To: c:\Users\Leandro\Documents\codigos\pos_graduacao\pos_graduacao_analise_preparacao_dados\data\DENGBR24.csv
100%|██████████| 1.84G/1.84G [00:42<00:00, 43.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1WXvMMrBsmnz8Z2x68fpS0T-o2d1qZxJ_
From (redirected): https://drive.google.com/uc?id=1WXvMMrBsmnz8Z2x68fpS0T-o2d1qZxJ_&confirm=t&uuid=9f98e0f6-f73f-48fd-8e74-2012696a4cfe
To: c:\Users\Leandro\Documents\codigos\pos_graduacao\pos_graduacao_analise_preparacao_dados\data\DENGBR25.csv
100%|██████████| 458M/458M [00:10<00:00, 42.9MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1IY0fqEHXum51sTRzfpoLpP1x6A9USz1w
To: c:\Users\Leandro\Documents\codigos\pos_graduacao\pos_graduacao_analise_preparacao_dados\data\DENGBR26.csv
100%|████████

In [3]:
df = pd.read_parquet(
    "data/dengue.parquet"
)

display(df.head())

with open("./data/municipios.json", "r", encoding="utf-8") as f:
    municipios = json.load(f)

with open("./data/estados.json", "r", encoding="utf-8") as f:
    estados = json.load(f)

estados_municipios = [
    {
        **m,
        "sigla": estados[str(m["COD UF"])]["sigla"],
        "nome": estados[str(m["COD UF"])]["nome"]
    }
    for m in municipios
    if str(m["COD UF"]) in estados
]

uf_siglas = {
    m["COD UF"]: m["sigla"]
    for m in estados_municipios
}

municipios_por_codigo = {
    int(str(m["COD"])[:-1]): m["NOME"]
    for m in estados_municipios
}

def map_uf(x):
    if pd.isna(x):
        return x
    try:
        return uf_siglas.get(int(x), x)
    except (ValueError, TypeError):
        return x

df['SG_UF_NOT'] = df['SG_UF_NOT'].apply(map_uf)

df['SG_UF'] = df['SG_UF'].apply(map_uf)

df['ID_MUNICIP'] = (
    df['ID_MUNICIP']
    .map(municipios_por_codigo)
    .fillna(df['ID_MUNICIP'])
)

df['ID_MUNICIP'] = df['ID_MUNICIP'].astype(str).replace('nan', None)

df['ID_MN_RESI'] = (
    df['ID_MN_RESI']
    .map(municipios_por_codigo)
    .fillna(df['ID_MN_RESI'])
)

df['ID_MN_RESI'] = df['ID_MN_RESI'].astype(str).replace('nan', None)

display(df.head())

caminho_limpo = "data/dengue_limpo.parquet"
df.to_parquet(caminho_limpo, index=False)

,TP_NOT,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,DT_SIN_PRI,SEM_PRI,ANO_NASC,NU_IDADE_N,CS_SEXO,CS_GESTANT,CS_RACA,CS_ESCOL_N,SG_UF,ID_MN_RESI,DT_INVEST,FEBRE,MIALGIA,CEFALEIA,EXANTEMA,VOMITO,NAUSEA,DOR_COSTAS,CONJUNTVIT,ARTRITE,ARTRALGIA,PETEQUIA_N,LEUCOPENIA,LACO,DOR_RETRO,DIABETES,HEMATOLOG,HEPATOPAT,RENAL,HIPERTENSA,ACIDO_PEPT,AUTO_IMUNE,DT_CHIK_S1,DT_CHIK_S2,DT_PRNT,RES_CHIKS1,RES_CHIKS2,RESUL_PRNT,DT_SORO,RESUL_SORO,DT_NS1,RESUL_NS1,DT_VIRAL,RESUL_VI_N,DT_PCR,RESUL_PCR_,SOROTIPO,HISTOPA_N,IMUNOH_N,HOSPITALIZ,DT_INTERNA,UF,MUNICIPIO,TPAUTOCTO,COUFINF,COPAISINF,COMUNINF,CLASSI_FIN,CRITERIO,DOENCA_TRA,CLINC_CHIK,EVOLUCAO,DT_OBITO,DT_ENCERRA,ALRM_HIPOT,ALRM_PLAQ,ALRM_VOM,ALRM_SANG,ALRM_HEMAT,ALRM_ABDOM,ALRM_LETAR,ALRM_HEPAT,ALRM_LIQ,DT_ALRM,GRAV_PULSO,GRAV_CONV,GRAV_ENCH,GRAV_INSUF,GRAV_TAQUI,GRAV_EXTRE,GRAV_HIPOT,GRAV_HEMAT,GRAV_MELEN,GRAV_METRO,GRAV_SANG,GRAV_AST,GRAV_MIOC,GRAV_CONSC,GRAV_ORGAO,DT_GRAV,MANI_HEMOR,EPISTAXE,GENGIVO,METRO,PETEQUIAS,HEMATURA,SANGRAM,LACO_N,PLASMATICO,EVIDENCIA,PLAQ_MENOR,CON_FHD,COMPLICA,TP_SISTEMA,NDUPLIC_N,DT_DIGITA,CS_FLXRET,FLXRECEBI,MIGRADO_W
0,2,2024-03-04,202410,2024,32,320150,2024-03-02,202409,2018.0,4005,F,6.0,1.0,10.0,32,320150.0,2023-03-24,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,None,4.0,None,4.0,None,4.0,2024-03-06,1.0,2.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,1.0,0.0,0.0,1.0,None,2024-04-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-24,0.0,0.0,0.0
1,2,2024-04-17,202416,2024,32,320465,2024-04-15,202416,2017.0,4006,M,6.0,4.0,10.0,32,320465.0,2023-03-29,1.0,1.0,1.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,2024-07-01,2.0,None,0.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,2.0,0.0,0.0,1.0,None,2024-07-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-29,0.0,0.0,0.0
2,2,2024-06-22,202425,2024,32,320520,2024-06-11,202424,2006.0,4018,M,6.0,4.0,5.0,32,320520.0,2023-06-22,1.0,1.0,1.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,2024-06-11,1.0,None,4.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,1.0,0.0,0.0,1.0,None,2023-07-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-06-22,0.0,0.0,0.0
3,2,2023-12-31,202401,2023,32,320020,2023-12-31,202401,1995.0,4028,F,6.0,1.0,9.0,32,320020.0,2023-12-31,1.0,2.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,None,4.0,None,4.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,11.0,2.0,0.0,1.0,1.0,None,2023-12-31,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2023-12-27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-12-31,0.0,0.0,0.0
4,2,2023-12-31,202401,2023,32,320130,2023-12-31,202401,1959.0,4064,M,6.0,4.0,9.0,32,320130.0,2023-12-31,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,None,4.0,None,4.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,2.0,0.0,1.0,1.0,None,2024-01-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-12-31,0.0,0.0,0.0


,TP_NOT,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,DT_SIN_PRI,SEM_PRI,ANO_NASC,NU_IDADE_N,CS_SEXO,CS_GESTANT,CS_RACA,CS_ESCOL_N,SG_UF,ID_MN_RESI,DT_INVEST,FEBRE,MIALGIA,CEFALEIA,EXANTEMA,VOMITO,NAUSEA,DOR_COSTAS,CONJUNTVIT,ARTRITE,ARTRALGIA,PETEQUIA_N,LEUCOPENIA,LACO,DOR_RETRO,DIABETES,HEMATOLOG,HEPATOPAT,RENAL,HIPERTENSA,ACIDO_PEPT,AUTO_IMUNE,DT_CHIK_S1,DT_CHIK_S2,DT_PRNT,RES_CHIKS1,RES_CHIKS2,RESUL_PRNT,DT_SORO,RESUL_SORO,DT_NS1,RESUL_NS1,DT_VIRAL,RESUL_VI_N,DT_PCR,RESUL_PCR_,SOROTIPO,HISTOPA_N,IMUNOH_N,HOSPITALIZ,DT_INTERNA,UF,MUNICIPIO,TPAUTOCTO,COUFINF,COPAISINF,COMUNINF,CLASSI_FIN,CRITERIO,DOENCA_TRA,CLINC_CHIK,EVOLUCAO,DT_OBITO,DT_ENCERRA,ALRM_HIPOT,ALRM_PLAQ,ALRM_VOM,ALRM_SANG,ALRM_HEMAT,ALRM_ABDOM,ALRM_LETAR,ALRM_HEPAT,ALRM_LIQ,DT_ALRM,GRAV_PULSO,GRAV_CONV,GRAV_ENCH,GRAV_INSUF,GRAV_TAQUI,GRAV_EXTRE,GRAV_HIPOT,GRAV_HEMAT,GRAV_MELEN,GRAV_METRO,GRAV_SANG,GRAV_AST,GRAV_MIOC,GRAV_CONSC,GRAV_ORGAO,DT_GRAV,MANI_HEMOR,EPISTAXE,GENGIVO,METRO,PETEQUIAS,HEMATURA,SANGRAM,LACO_N,PLASMATICO,EVIDENCIA,PLAQ_MENOR,CON_FHD,COMPLICA,TP_SISTEMA,NDUPLIC_N,DT_DIGITA,CS_FLXRET,FLXRECEBI,MIGRADO_W
0,2,2024-03-04,202410,2024,ES,Colatina,2024-03-02,202409,2018.0,4005,F,6.0,1.0,10.0,ES,Colatina,2023-03-24,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,None,4.0,None,4.0,None,4.0,2024-03-06,1.0,2.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,1.0,0.0,0.0,1.0,None,2024-04-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-24,0.0,0.0,0.0
1,2,2024-04-17,202416,2024,ES,São domingos do Norte,2024-04-15,202416,2017.0,4006,M,6.0,4.0,10.0,ES,São domingos do Norte,2023-03-29,1.0,1.0,1.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,2024-07-01,2.0,None,0.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,2.0,0.0,0.0,1.0,None,2024-07-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-29,0.0,0.0,0.0
2,2,2024-06-22,202425,2024,ES,Vila Velha,2024-06-11,202424,2006.0,4018,M,6.0,4.0,5.0,ES,Vila Velha,2023-06-22,1.0,1.0,1.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,2024-06-11,1.0,None,4.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,1.0,0.0,0.0,1.0,None,2023-07-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-06-22,0.0,0.0,0.0
3,2,2023-12-31,202401,2023,ES,Alegre,2023-12-31,202401,1995.0,4028,F,6.0,1.0,9.0,ES,Alegre,2023-12-31,1.0,2.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,None,4.0,None,4.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,11.0,2.0,0.0,1.0,1.0,None,2023-12-31,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2023-12-27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-12-31,0.0,0.0,0.0
4,2,2023-12-31,202401,2023,ES,Cariacica,2023-12-31,202401,1959.0,4064,M,6.0,4.0,9.0,ES,Cariacica,2023-12-31,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,None,None,None,0.0,0.0,0.0,None,4.0,None,4.0,None,4.0,None,4.0,0.0,0.0,0.0,2.0,None,0.0,0.0,1.0,0.0,0.0,0.0,10.0,2.0,0.0,1.0,1.0,None,2024-01-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-12-31,0.0,0.0,0.0


In [4]:
df = spark.read.parquet('data/dengue_limpo.parquet')

for coluna in df.columns:
    novo_nome = coluna.strip().upper()

    if coluna != novo_nome:
        df = df.withColumnRenamed(
            coluna, novo_nome
        )

print("Número de registros:", df.count())
print("Número de variáveis:", len(df.columns))

df.printSchema()

df.show(5, truncate=False)

Número de registros: 8521502
Número de variáveis: 115
root
 |-- TP_NOT: integer (nullable = true)
 |-- DT_NOTIFIC: date (nullable = true)
 |-- SEM_NOT: integer (nullable = true)
 |-- NU_ANO: integer (nullable = true)
 |-- SG_UF_NOT: string (nullable = true)
 |-- ID_MUNICIP: string (nullable = true)
 |-- DT_SIN_PRI: date (nullable = true)
 |-- SEM_PRI: integer (nullable = true)
 |-- ANO_NASC: double (nullable = true)
 |-- NU_IDADE_N: integer (nullable = true)
 |-- CS_SEXO: string (nullable = true)
 |-- CS_GESTANT: double (nullable = true)
 |-- CS_RACA: double (nullable = true)
 |-- CS_ESCOL_N: double (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- ID_MN_RESI: string (nullable = true)
 |-- DT_INVEST: date (nullable = true)
 |-- FEBRE: double (nullable = true)
 |-- MIALGIA: double (nullable = true)
 |-- CEFALEIA: double (nullable = true)
 |-- EXANTEMA: double (nullable = true)
 |-- VOMITO: double (nullable = true)
 |-- NAUSEA: double (nullable = true)
 |-- DOR_COSTAS: double (

In [5]:
print("VARIÁVEIS DISPONÍVEIS")

for coluna in sorted(df.columns):
    print(coluna)

VARIÁVEIS DISPONÍVEIS
ACIDO_PEPT
ALRM_ABDOM
ALRM_HEMAT
ALRM_HEPAT
ALRM_HIPOT
ALRM_LETAR
ALRM_LIQ
ALRM_PLAQ
ALRM_SANG
ALRM_VOM
ANO_NASC
ARTRALGIA
ARTRITE
AUTO_IMUNE
CEFALEIA
CLASSI_FIN
CLINC_CHIK
COMPLICA
COMUNINF
CONJUNTVIT
CON_FHD
COPAISINF
COUFINF
CRITERIO
CS_ESCOL_N
CS_FLXRET
CS_GESTANT
CS_RACA
CS_SEXO
DIABETES
DOENCA_TRA
DOR_COSTAS
DOR_RETRO
DT_ALRM
DT_CHIK_S1
DT_CHIK_S2
DT_DIGITA
DT_ENCERRA
DT_GRAV
DT_INTERNA
DT_INVEST
DT_NOTIFIC
DT_NS1
DT_OBITO
DT_PCR
DT_PRNT
DT_SIN_PRI
DT_SORO
DT_VIRAL
EPISTAXE
EVIDENCIA
EVOLUCAO
EXANTEMA
FEBRE
FLXRECEBI
GENGIVO
GRAV_AST
GRAV_CONSC
GRAV_CONV
GRAV_ENCH
GRAV_EXTRE
GRAV_HEMAT
GRAV_HIPOT
GRAV_INSUF
GRAV_MELEN
GRAV_METRO
GRAV_MIOC
GRAV_ORGAO
GRAV_PULSO
GRAV_SANG
GRAV_TAQUI
HEMATOLOG
HEMATURA
HEPATOPAT
HIPERTENSA
HISTOPA_N
HOSPITALIZ
ID_MN_RESI
ID_MUNICIP
IMUNOH_N
LACO
LACO_N
LEUCOPENIA
MANI_HEMOR
METRO
MIALGIA
MIGRADO_W
MUNICIPIO
NAUSEA
NDUPLIC_N
NU_ANO
NU_IDADE_N
PETEQUIAS
PETEQUIA_N
PLAQ_MENOR
PLASMATICO
RENAL
RESUL_NS1
RESUL_PCR_
RESUL_PRNT
RESUL_

In [6]:
ALVO_ORIGINAL = "CLASSI_FIN"

if ALVO_ORIGINAL not in df.columns:
    raise ValueError(
        "CLASSI_FIN não encontrada. "
        "Verifique o esquema e o dicionário da base."
    )

df.groupBy("CLASSI_FIN").count().orderBy(
    "CLASSI_FIN"
).show(50, truncate=False)

df = df.withColumn(
    "CLASSI_COD",
    F.col("CLASSI_FIN").cast("float").cast("int").cast("string")
)

df_modelo = (
    df.filter(
        F.col("CLASSI_COD").isin("10", "11", "12")
    )
    .withColumn(
        "gravidade",
        F.when(
            F.col("CLASSI_COD").isin("11", "12"),
            F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )
)

df_modelo.groupBy("gravidade").count().show()

+----------+-------+
|CLASSI_FIN|count  |
+----------+-------+
|NULL      |111277 |
|0.0       |5731   |
|8.0       |731905 |
|10.0      |7518484|
|11.0      |142175 |
|12.0      |11930  |
+----------+-------+

+---------+-------+
|gravidade|  count|
+---------+-------+
|      0.0|7518484|
|      1.0| 154105|
+---------+-------+



In [7]:
# Características demográficas
demograficas = [
    "CS_SEXO",
    "CS_RACA",
    "CS_GESTANT"
]

# Sintomas e manifestações clínicas candidatas
clinicas = [
    "FEBRE",
    "MIALGIA",
    "CEFALEIA",
    "EXANTEMA",
    "VOMITO",
    "NAUSEA",
    "DOR_COSTAS",
    "CONJUNTVIT",
    "ARTRALGIA",
    "ARTRITE",
    "DOR_RETRO"
]

# Comorbidades
comorbidades = [
    "DIABETES",
    "HEMATOLOG",
    "HEPATOPAT",
    "RENAL",
    "HIPERTENSA",
    "ACIDO_PEPT",
    "AUTO_IMUNE"
]

# Contexto epidemiológico
epidemiologicas = [
    "SG_UF",
    "SEM_NOT"
]

# True: usar somente idade e variáveis
# demográficas/epidemiológicas.
# False: acrescentar sintomas e comorbidades.
MODO_ESTRITO = False

candidatas = demograficas + epidemiologicas

if not MODO_ESTRITO:
    candidatas += clinicas + comorbidades

variaveis = [
    c for c in candidatas
    if c in df_modelo.columns
]

print("Variáveis incluídas:", variaveis)

print(
    "Variáveis indisponíveis:",
    sorted(set(candidatas) - set(variaveis))
)

if not variaveis:
    raise ValueError("Nenhum preditor encontrado.")

Variáveis incluídas: ['CS_SEXO', 'CS_RACA', 'CS_GESTANT', 'SG_UF', 'SEM_NOT', 'FEBRE', 'MIALGIA', 'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA', 'DOR_COSTAS', 'CONJUNTVIT', 'ARTRALGIA', 'ARTRITE', 'DOR_RETRO', 'DIABETES', 'HEMATOLOG', 'HEPATOPAT', 'RENAL', 'HIPERTENSA', 'ACIDO_PEPT', 'AUTO_IMUNE']
Variáveis indisponíveis: []


In [8]:
if "NU_IDADE_N" in df_modelo.columns:

    codigo = F.col("NU_IDADE_N").cast("int")
    unidade = F.floor(codigo / 1000)
    valor = F.pmod(codigo, 1000)

    idade = (
        F.when(unidade == 4, valor.cast("double"))
         .when(unidade == 3, valor / F.lit(12.0))
         .when(unidade == 2, valor / F.lit(365.25))
         .when(unidade == 1, valor / F.lit(8766.0))
    )

    df_modelo = df_modelo.withColumn(
        "IDADE_ANOS",
        F.when(
            (idade >= 0) & (idade <= 120),
            idade
        ).otherwise(None)
    )

    variaveis_numericas = ["IDADE_ANOS"]

else:
    variaveis_numericas = []

print("Variáveis numéricas:", variaveis_numericas)


Variáveis numéricas: ['IDADE_ANOS']


In [9]:
categoricas = list(dict.fromkeys(variaveis))

for c in categoricas:
    df_modelo = df_modelo.withColumn(
        c,
        F.when(
            F.col(c).isNull() |
            F.trim(F.col(c).cast("string")).isin(
                "", "null", "None"
            ),
            F.lit("Ignorado")
        ).otherwise(
            F.trim(F.col(c).cast("string"))
        )
    )

# Manter apenas campos necessários ao modelo
colunas_finais = (
    ["gravidade"] +
    categoricas +
    variaveis_numericas
)

colunas_faltantes = [c for c in colunas_finais if c not in df_modelo.columns]

print("Colunas que não existem no DataFrame:", colunas_faltantes)

dados = df_modelo.select(
    *list(dict.fromkeys(colunas_finais))
)

dados.show(5, truncate=False)

Colunas que não existem no DataFrame: []
+---------+-------+-------+----------+-----+-------+-----+-------+--------+--------+------+------+----------+----------+---------+-------+---------+--------+---------+---------+-----+----------+----------+----------+----------+
|gravidade|CS_SEXO|CS_RACA|CS_GESTANT|SG_UF|SEM_NOT|FEBRE|MIALGIA|CEFALEIA|EXANTEMA|VOMITO|NAUSEA|DOR_COSTAS|CONJUNTVIT|ARTRALGIA|ARTRITE|DOR_RETRO|DIABETES|HEMATOLOG|HEPATOPAT|RENAL|HIPERTENSA|ACIDO_PEPT|AUTO_IMUNE|IDADE_ANOS|
+---------+-------+-------+----------+-----+-------+-----+-------+--------+--------+------+------+----------+----------+---------+-------+---------+--------+---------+---------+-----+----------+----------+----------+----------+
|0.0      |F      |1.0    |6.0       |ES   |202410 |1.0  |1.0    |1.0     |2.0     |2.0   |2.0   |2.0       |2.0       |2.0      |2.0    |2.0      |2.0     |2.0      |2.0      |2.0  |2.0       |2.0       |2.0       |5.0       |
|0.0      |M      |4.0    |6.0       |ES   |202

In [10]:
for variavel in ["CS_SEXO", "SG_UF"]:

    if variavel in dados.columns:

        resultado = (
            dados.groupBy(variavel)
            .agg(
                F.count("*").alias("total"),
                F.avg("gravidade").alias("proporcao_grave")
            )
            .orderBy(F.desc("total"))
        )

        print("\nAnálise:", variavel)
        resultado.show(30, truncate=False)


Análise: CS_SEXO
+--------+-------+--------------------+
|CS_SEXO |total  |proporcao_grave     |
+--------+-------+--------------------+
|F       |4182964|0.020559823130201455|
|M       |3481303|0.019532916267271192|
|I       |8312   |0.01239172281039461 |
|Ignorado|10     |0.1                 |
+--------+-------+--------------------+


Análise: SG_UF
+-----+-------+--------------------+
|SG_UF|total  |proporcao_grave     |
+-----+-------+--------------------+
|SP   |3077735|0.015894805758130574|
|MG   |1555770|0.013287953874930099|
|PR   |721162 |0.024880678682459696|
|GO   |485322 |0.0314554048652235  |
|SC   |342799 |0.02032094609377507 |
|RJ   |292287 |0.020168532982992742|
|RS   |264477 |0.015415329121246837|
|DF   |205899 |0.05995172390346724 |
|BA   |193229 |0.041148067836608376|
|ES   |177724 |0.020357408115955076|
|MT   |77014  |0.024242345547562782|
|PA   |37139  |0.02959153450550634 |
|PE   |33488  |0.029951027233635928|
|MS   |27123  |0.047229288795487226|
|AL   |25662  |0

In [11]:
treino, validacao, teste = dados.randomSplit(
    [0.70, 0.15, 0.15],
    seed=42
)

treino = treino.cache()
validacao = validacao.cache()
teste = teste.cache()

for nome, parte in [
    ("Treino", treino),
    ("Validação", validacao),
    ("Teste", teste)
]:
    print(nome)
    parte.groupBy("gravidade").count().show()

    if parte.select("gravidade").distinct().count() < 2:
        raise ValueError(
            f"{nome} não contém ambas as classes."
        )


Treino
+---------+-------+
|gravidade|  count|
+---------+-------+
|      0.0|5265097|
|      1.0| 107769|
+---------+-------+

Validação
+---------+-------+
|gravidade|  count|
+---------+-------+
|      0.0|1126392|
|      1.0|  23082|
+---------+-------+

Teste
+---------+-------+
|gravidade|  count|
+---------+-------+
|      0.0|1126995|
|      1.0|  23254|
+---------+-------+



In [ ]:
etapas = []

indexadas = []

for c in categoricas:
    saida = c + "_IDX"

    etapas.append(
        StringIndexer(
            inputCol=c,
            outputCol=saida,
            handleInvalid="keep"
        )
    )

    indexadas.append(saida)

if indexadas:
    codificadas = [
        c + "_OHE" for c in categoricas
    ]

    etapas.append(
        OneHotEncoder(
            inputCols=indexadas,
            outputCols=codificadas,
            handleInvalid="keep"
        )
    )
else:
    codificadas = []

numericas_tratadas = []

if variaveis_numericas:

    numericas_tratadas = [
        c + "_IMP" for c in variaveis_numericas
    ]

    etapas.append(
        Imputer(
            inputCols=variaveis_numericas,
            outputCols=numericas_tratadas,
            strategy="median"
        )
    )

entradas = codificadas + numericas_tratadas

etapas.append(
    VectorAssembler(
        inputCols=entradas,
        outputCol="features",
        handleInvalid="keep"
    )
)

pipeline_preparo = Pipeline(stages=etapas)

preparador = pipeline_preparo.fit(treino)

treino_ml = preparador.transform(treino)
validacao_ml = preparador.transform(validacao)
teste_ml = preparador.transform(teste)

print("Pré-processamento concluído.")

Pré-processamento concluído.


In [13]:
contagens = {
    float(r["gravidade"]): r["count"]
    for r in treino_ml.groupBy("gravidade").count().collect()
}

n0 = contagens.get(0.0, 0)
n1 = contagens.get(1.0, 0)

if n0 == 0 or n1 == 0:
    raise ValueError("Uma das classes está ausente.")

peso_positivo = min(n0 / n1, 30.0)

treino_ml = treino_ml.withColumn(
    "peso_classe",
    F.when(
        F.col("gravidade") == 1.0,
        F.lit(peso_positivo)
    ).otherwise(F.lit(1.0))
)

print("Peso da classe positiva:", peso_positivo)

Peso da classe positiva: 30.0
